In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [12]:
df1 = pd.read_csv("C:/Users/Administrator/Desktop/Travel_Productioniztion/Travel_Dataset/raw_datasets/flights.csv")
df1.head(3)

,travelCode,userCode,from,to,flightType,price,time,distance,agency,date
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,09/26/2019
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,09/30/2019
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,10/03/2019


In [13]:
df2 = pd.read_csv('C:/Users/Administrator/Desktop/Travel_Productioniztion/Travel_Dataset/raw_datasets/hotels.csv')
df2.head()

,travelCode,userCode,name,place,days,price,total,date
0,0,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
1,2,0,Hotel K,Salvador (BH),2,263.41,526.82,10/10/2019
2,7,0,Hotel K,Salvador (BH),3,263.41,790.23,11/14/2019
3,11,0,Hotel K,Salvador (BH),4,263.41,1053.64,12/12/2019
4,13,0,Hotel A,Florianopolis (SC),1,313.02,313.02,12/26/2019


In [15]:
df3 = pd.read_csv('C:/Users/Administrator/Desktop/Travel_Productioniztion/Travel_Dataset/raw_datasets/users.csv')
df3.head()

,code,company,name,gender,age
0,0,4You,Roy Braun,male,21
1,1,4You,Joseph Holsten,male,37
2,2,4You,Wilma Mcinnis,female,48
3,3,4You,Paula Daniel,female,23
4,4,4You,Patricia Carson,female,44


In [16]:
df_flight_hotel = pd.merge(df1, df2, on='travelCode', how='inner')
df_flight_hotel.head(3)

,travelCode,userCode_x,from,to,flightType,price_x,time,distance,agency,date_x,userCode_y,name,place,days,price_y,total,date_y
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,09/26/2019,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,09/30/2019,0,Hotel A,Florianopolis (SC),4,313.02,1252.08,09/26/2019
2,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,10/10/2019,0,Hotel K,Salvador (BH),2,263.41,526.82,10/10/2019


In [17]:
final_df = pd.merge(df_flight_hotel, df3,left_on='userCode_x',right_on='code',how='inner')
final_df.head()

,travelCode,userCode_x,from,to,flightType,price_x,time,distance,agency,date_x,...,place,days,price_y,total,date_y,code,company,name_y,gender,age
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,09/26/2019,...,Florianopolis (SC),4,313.02,1252.08,09/26/2019,0,4You,Roy Braun,male,21
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,09/30/2019,...,Florianopolis (SC),4,313.02,1252.08,09/26/2019,0,4You,Roy Braun,male,21
2,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,10/10/2019,...,Salvador (BH),2,263.41,526.82,10/10/2019,0,4You,Roy Braun,male,21
3,2,0,Salvador (BH),Aracaju (SE),firstClass,1531.92,2.16,830.86,CloudFy,10/12/2019,...,Salvador (BH),2,263.41,526.82,10/10/2019,0,4You,Roy Braun,male,21
4,7,0,Aracaju (SE),Salvador (BH),economic,964.83,2.16,830.86,CloudFy,11/14/2019,...,Salvador (BH),3,263.41,790.23,11/14/2019,0,4You,Roy Braun,male,21


In [18]:
final_df.drop(columns=['userCode_y', 'code'], inplace=True)
final_df.rename(columns={'userCode_x': 'userCode'}, inplace=True)

In [19]:
final_df.rename(columns={
    'price_x': 'flight_price',
    'price_y': 'hotel_price_per_day',
    'name_x': 'hotel_name',
    'name_y': 'user_name',
    'date_x': 'flight_date',
    'date_y': 'hotel_date'
}, inplace=True)

In [20]:
final_df.shape

(81104, 20)

In [21]:
final_df['flight_date'] = pd.to_datetime(final_df['flight_date'])
final_df['hotel_date'] = pd.to_datetime(final_df['hotel_date'])

In [22]:
final_df['trip_gap_days'] = (final_df['hotel_date'] - final_df['flight_date']).dt.days

In [24]:
final_df['trip_gap_days'] = (final_df['hotel_date'] - final_df['flight_date']).dt.days.abs()

In [25]:
final_df['total_trip_cost'] = final_df['flight_price'] + final_df['total']

In [26]:
final_df.groupby('to')['total_trip_cost'].mean().sort_values(ascending=False)

to
Salvador (BH)          1926.636376
Florianopolis (SC)     1747.481382
Aracaju (SE)           1602.292514
Recife (PE)            1601.780651
Natal (RN)             1411.677750
Brasilia (DF)          1341.660109
Rio de Janeiro (RJ)    1263.842000
Campo Grande (MS)      1244.427030
Sao Paulo (SP)         1063.722482
Name: total_trip_cost, dtype: float64

In [27]:
final_df.groupby('age')['total_trip_cost'].mean()

age
21    1473.172966
22    1496.951815
23    1487.272014
24    1487.506197
25    1460.158858
26    1523.160611
27    1535.060994
28    1504.435224
29    1492.786206
30    1469.213225
31    1488.259181
32    1473.826189
33    1494.960715
34    1488.206654
35    1508.781135
36    1513.515322
37    1521.498204
38    1556.458262
39    1493.692634
40    1512.537244
41    1500.810367
42    1490.431550
43    1483.522313
44    1479.044947
45    1487.643184
46    1487.448585
47    1505.321184
48    1487.827074
49    1467.162439
50    1505.869979
51    1501.186057
52    1457.386213
53    1500.639505
54    1456.855354
55    1483.421947
56    1512.752151
57    1456.381527
58    1520.119078
59    1510.924948
60    1495.376309
61    1472.846089
62    1502.170934
63    1468.192753
64    1492.603449
65    1485.168268
Name: total_trip_cost, dtype: float64

In [28]:
final_df.groupby('days')['total_trip_cost'].mean()

days
1    1171.854394
2    1388.245909
3    1598.677165
4    1817.310722
Name: total_trip_cost, dtype: float64

In [29]:
final_df.groupby('agency')['total_trip_cost'].sum().sort_values(ascending=False)

agency
Rainbow        50759372.53
CloudFy        50559100.05
FlyingDrops    19847237.07
Name: total_trip_cost, dtype: float64

In [30]:
final_df.head(4)

,travelCode,userCode,from,to,flightType,flight_price,time,distance,agency,flight_date,...,days,hotel_price_per_day,total,hotel_date,company,user_name,gender,age,trip_gap_days,total_trip_cost
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019-09-26,...,4,313.02,1252.08,2019-09-26,4You,Roy Braun,male,21,0,2686.46
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019-09-30,...,4,313.02,1252.08,2019-09-26,4You,Roy Braun,male,21,4,2544.37
2,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,2019-10-10,...,2,263.41,526.82,2019-10-10,4You,Roy Braun,male,21,0,2210.87
3,2,0,Salvador (BH),Aracaju (SE),firstClass,1531.92,2.16,830.86,CloudFy,2019-10-12,...,2,263.41,526.82,2019-10-10,4You,Roy Braun,male,21,2,2058.74


In [31]:
final_df.isnull().sum()

travelCode             0
userCode               0
from                   0
to                     0
flightType             0
flight_price           0
time                   0
distance               0
agency                 0
flight_date            0
hotel_name             0
place                  0
days                   0
hotel_price_per_day    0
total                  0
hotel_date             0
company                0
user_name              0
gender                 0
age                    0
trip_gap_days          0
total_trip_cost        0
dtype: int64

In [32]:
final_df = final_df.drop_duplicates(subset='travelCode')

In [33]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40552 entries, 0 to 81102
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   travelCode           40552 non-null  int64         
 1   userCode             40552 non-null  int64         
 2   from                 40552 non-null  str           
 3   to                   40552 non-null  str           
 4   flightType           40552 non-null  str           
 5   flight_price         40552 non-null  float64       
 6   time                 40552 non-null  float64       
 7   distance             40552 non-null  float64       
 8   agency               40552 non-null  str           
 9   flight_date          40552 non-null  datetime64[us]
 10  hotel_name           40552 non-null  str           
 11  place                40552 non-null  str           
 12  days                 40552 non-null  int64         
 13  hotel_price_per_day  40552 non-null  float

In [34]:
ml_df = final_df.drop(columns=[
    'travelCode',
    'userCode',
    'user_name',
    'hotel_name',
    'total',
    'flight_date',
    'hotel_date'
])

In [35]:
ml_df.head()

,from,to,flightType,flight_price,time,distance,agency,place,days,hotel_price_per_day,company,gender,age,trip_gap_days,total_trip_cost
0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,Florianopolis (SC),4,313.02,4You,male,21,0,2686.46
2,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,Salvador (BH),2,263.41,4You,male,21,0,2210.87
4,Aracaju (SE),Salvador (BH),economic,964.83,2.16,830.86,CloudFy,Salvador (BH),3,263.41,4You,male,21,0,1755.06
6,Brasilia (DF),Salvador (BH),premium,1268.97,1.76,676.56,Rainbow,Salvador (BH),4,263.41,4You,male,21,0,2322.61
8,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,Florianopolis (SC),1,313.02,4You,male,21,0,1747.40


In [36]:
le = LabelEncoder()
ml_df['gender'] = le.fit_transform(ml_df['gender'])

In [37]:
ml_df['gender'].value_counts()

gender
0    13690
1    13585
2    13277
Name: count, dtype: int64

In [38]:
final_df['gender'].value_counts()

gender
female    13690
male      13585
none      13277
Name: count, dtype: int64

In [39]:
ml_df = pd.get_dummies(ml_df, drop_first=True)

In [40]:
ml_df.head()

,flight_price,time,distance,days,hotel_price_per_day,gender,age,trip_gap_days,total_trip_cost,from_Brasilia (DF),...,place_Florianopolis (SC),place_Natal (RN),place_Recife (PE),place_Rio de Janeiro (RJ),place_Salvador (BH),place_Sao Paulo (SP),company_Acme Factory,company_Monsters CYA,company_Umbrella LTDA,company_Wonka Company
0,1434.38,1.76,676.53,4,313.02,1,21,0,2686.46,False,...,True,False,False,False,False,False,False,False,False,False
2,1684.05,2.16,830.86,2,263.41,1,21,0,2210.87,False,...,False,False,False,False,True,False,False,False,False,False
4,964.83,2.16,830.86,3,263.41,1,21,0,1755.06,False,...,False,False,False,False,True,False,False,False,False,False
6,1268.97,1.76,676.56,4,263.41,1,21,0,2322.61,True,...,False,False,False,False,True,False,False,False,False,False
8,1434.38,1.76,676.53,1,313.02,1,21,0,1747.40,False,...,True,False,False,False,False,False,False,False,False,False


In [41]:
bool_cols = ml_df.select_dtypes(include='bool').columns
ml_df[bool_cols] = ml_df[bool_cols].astype(int)

In [42]:
ml_df

,flight_price,time,distance,days,hotel_price_per_day,gender,age,trip_gap_days,total_trip_cost,from_Brasilia (DF),...,place_Florianopolis (SC),place_Natal (RN),place_Recife (PE),place_Rio de Janeiro (RJ),place_Salvador (BH),place_Sao Paulo (SP),company_Acme Factory,company_Monsters CYA,company_Umbrella LTDA,company_Wonka Company
0,1434.38,1.76,676.53,4,313.02,1,21,0,2686.46,0,...,1,0,0,0,0,0,0,0,0,0
2,1684.05,2.16,830.86,2,263.41,1,21,0,2210.87,0,...,0,0,0,0,1,0,0,0,0,0
4,964.83,2.16,830.86,3,263.41,1,21,0,1755.06,0,...,0,0,0,0,1,0,0,0,0,0
6,1268.97,1.76,676.56,4,263.41,1,21,0,2322.61,1,...,0,0,0,0,1,0,0,0,0,0
8,1434.38,1.76,676.53,1,313.02,1,21,0,1747.40,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81094,885.37,1.66,637.56,3,247.62,1,35,0,1628.23,0,...,0,0,0,0,0,0,0,0,1,0
81096,814.55,1.66,637.56,1,247.62,1,35,0,1062.17,0,...,0,0,0,0,0,0,0,0,1,0
81098,857.32,1.49,573.81,3,60.39,1,35,0,1038.49,0,...,0,0,0,0,0,0,0,0,1,0
81100,949.58,1.49,573.81,3,60.39,1,35,0,1130.75,0,...,0,0,0,0,0,0,0,0,1,0


In [46]:
ml_df.to_csv(r'C:\Users\Administrator\Desktop\Travel_Productioniztion\Travel_Dataset\cleaned_dataset_for_ml\travel_ml_dataset_cleaned.csv', index=False)

In [45]:
import os
os.getcwd()

'c:\\Users\\Administrator\\Desktop\\Travel_Productioniztion\\notebooks'